# [Chapter 14 — Vector-Borne Diseases, §14.3] The Vector-Host $SIR$ Model and Ross-Macdonald $\mathcal{R}_0$

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 14 — Vector-Borne Diseases
**Considerations developed:** 4 (force-of-infection direction), 5 (linearization fidelity)
**Estimated runtime:** ≤ 30 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_14_vector_borne/01_vector_host_SIR.ipynb)


## What this notebook does

Implements the classic Ross-Macdonald vector-host model for an arbovirus (parameters loosely calibrated to dengue). The model has two coupled compartmental systems: human ($S_H, I_H, R_H$) and vector ($S_V, I_V$, with no recovery — vectors die infected). The notebook compares the simulated invasion phase to the closed-form Ross-Macdonald $\mathcal{R}_0$ and confirms the threshold prediction.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_14
from shared.verification import assert_within_tolerance
set_seed_chapter_14()
book_style()


## Vector-host system

$$\dot S_H = -a\,b\,(I_V/N) S_H, \qquad \dot I_H = +a\,b\,(I_V/N) S_H - \gamma_H I_H,$$
$$\dot S_V = \mu_v - a\,c\,S_V (I_H/N) - \mu_v S_V, \qquad \dot I_V = a\,c\,S_V (I_H/N) - \mu_v I_V.$$
Vector population is held at carrying capacity by birth = death; $S_V + I_V = 1$ (per unit vector population).
Then
$$\mathcal{R}_0 = \sqrt{\frac{a^2 b c (M/N)}{\mu_v \gamma_h}}$$
with vector-to-host ratio $M/N$.

In [ ]:
from shared.parameters import baseline_chapter_14
import math
p = baseline_chapter_14()
MN = p['M_over_N']; a = p['a']; b = p['b']; c = p['c']
mu_v = p['mu_v']; gamma_h = p['gamma_h']
R0 = math.sqrt(a**2 * b * c * MN / (mu_v * gamma_h))
print(f'Ross-Macdonald R_0 = {R0:.3f}')

def vh_rhs(y, t, p):
    SH, IH, RH, SV, IV = y
    a, b, c = p['a'], p['b'], p['c']
    mu_v, gamma_h = p['mu_v'], p['gamma_h']
    MN = p['M_over_N']
    # human FoI: per-host bites a*MN per day, fraction infectious IV, transmission b
    Lam_h = a * b * MN * IV
    # vector FoI: each vector bites a humans/day, frac infectious IH/N, transmission c
    Lam_v = a * c * IH
    return [-Lam_h*SH, Lam_h*SH - gamma_h*IH, gamma_h*IH,
            mu_v*(1-SV) - Lam_v*SV, Lam_v*SV - mu_v*IV]

y0 = [p['SH0'], p['IH0'], p['RH0'], p['SV0'], p['IV0']]
t = np.linspace(p['t_start'], p['t_end'], p['n_points'])
sol = odeint(vh_rhs, y0, t, args=(p,))
SH, IH, RH, SV, IV = sol.T


## Trajectory: human and vector prevalence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(t, IH, color=BOOK_COLORS['infectious'], lw=2)
axes[0].set_xlabel('time (days)'); axes[0].set_ylabel('I_H'); axes[0].set_title('Human infectious')
axes[0].grid(True, alpha=0.3)
axes[1].plot(t, IV, color=BOOK_COLORS['secondary'], lw=2)
axes[1].set_xlabel('time (days)'); axes[1].set_ylabel('I_V/M'); axes[1].set_title('Vector infectious fraction')
axes[1].grid(True, alpha=0.3)
fig.suptitle(f'Vector-host SIR, R_0 = {R0:.2f}')
fig.tight_layout()
plt.show()

att = float(RH[-1] + IH[-1])
print(f'Final human attack rate: {att:.3f}')
if R0 > 1:
    assert att > 0.05, 'Above threshold: significant attack rate expected'
    print('Verified: super-threshold parameters produce sustained outbreak.')


## Threshold sweep

Vary the vector-to-host ratio $M/N$ to traverse the threshold and confirm the closed-form prediction.

In [ ]:
MN_grid = np.linspace(0.5, 10, 25)
R0_curve = np.array([math.sqrt(a**2 * b * c * mn / (mu_v * gamma_h)) for mn in MN_grid])
att_curve = []
for mn in MN_grid:
    p2 = dict(p); p2['M_over_N'] = mn
    sol2 = odeint(vh_rhs, y0, t, args=(p2,))
    att_curve.append(float(sol2[-1, 2] + sol2[-1, 1]))
att_curve = np.array(att_curve)

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(MN_grid, R0_curve, color=BOOK_COLORS['primary'], lw=2, label='R_0 (closed form)')
ax1.axhline(1, ls=':', color='gray')
ax1.set_xlabel('vector-to-host ratio M/N')
ax1.set_ylabel('R_0', color=BOOK_COLORS['primary'])
ax2 = ax1.twinx()
ax2.plot(MN_grid, att_curve, 'o-', color=BOOK_COLORS['secondary'], lw=1.5,
         label='final attack rate (simulated)')
ax2.set_ylabel('final attack rate', color=BOOK_COLORS['secondary'])
ax1.set_title('Threshold behavior in vector-host model')
fig.tight_layout()
plt.show()

# Verify: attack rate should rise sharply when R_0 crosses 1
below = att_curve[R0_curve < 1].mean() if (R0_curve < 1).any() else 0
above = att_curve[R0_curve > 1.5].mean() if (R0_curve > 1.5).any() else 0
print(f'Mean attack rate below threshold (R_0<1): {below:.3f}')
print(f'Mean attack rate above threshold (R_0>1.5): {above:.3f}')
assert above > below, 'attack rate must increase with R_0'


## What's next

- Notebook 02 introduces seasonal vector dynamics (time-varying $\mu_v(t)$).
- Chapter 18 (Dengue case study) calibrates this model to real outbreak data.
- Vector-control interventions translate to multiplicative reductions in $a$, $\mu_v^{-1}$, or $M/N$ — Chapter 10's normalized-sensitivity machinery quantifies which lever is most effective.